In [ ]:
# 📊 Analyse des Tendances Mode Instagram

Ce notebook analyse les tendances de mode sur Instagram en utilisant l'API officielle.

**Données collectées (conformes aux règles de confidentialité):**
- Métadonnées des hashtags publics
- Volumes de mentions agrégés
- Analyse textuelle des descriptions
- Signaux statistiques et tendances

**⚠️ Note:** Ce notebook utilise l'API Instagram Graph officielle. Vous devez avoir un compte Business/Creator et un token d'accès valide.

In [ ]:
# Installation des dépendances
# !pip install requests pandas numpy matplotlib seaborn wordcloud textblob python-dotenv

In [ ]:
# Imports
import requests
import pandas as pd
import numpy as np
from datetime import datetime, timedelta
from collections import Counter
import matplotlib.pyplot as plt
import seaborn as sns
from wordcloud import WordCloud
from textblob import TextBlob
import json
import os
import re
from typing import Dict, List, Optional
from dataclasses import dataclass, asdict
import time

# Configuration visualisation
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette("husl")
pd.set_option('display.max_columns', None)

print("✅ Imports chargés avec succès")

## 1. Configuration API Instagram

Configuration du client pour l'API Instagram Graph officielle.

In [ ]:
# Configuration API Instagram Graph
# Remplacez par vos credentials

INSTAGRAM_ACCESS_TOKEN = os.getenv("INSTAGRAM_ACCESS_TOKEN", "YOUR_ACCESS_TOKEN")
INSTAGRAM_BUSINESS_ID = os.getenv("INSTAGRAM_BUSINESS_ID", "YOUR_BUSINESS_ID")

API_BASE_URL = "https://graph.facebook.com/v18.0"

class InstagramAPIClient:
    """Client pour l'API Instagram Graph officielle."""
    
    def __init__(self, access_token: str, business_id: str):
        self.access_token = access_token
        self.business_id = business_id
        self.base_url = API_BASE_URL
        self.rate_limit_remaining = 200
        
    def _make_request(self, endpoint: str, params: dict = None) -> dict:
        """Effectue une requête à l'API avec gestion des erreurs."""
        if params is None:
            params = {}
        params["access_token"] = self.access_token
        
        url = f"{self.base_url}/{endpoint}"
        
        try:
            response = requests.get(url, params=params, timeout=30)
            response.raise_for_status()
            
            # Mise à jour du rate limit si présent dans les headers
            if "x-business-use-case-usage" in response.headers:
                usage_info = json.loads(response.headers["x-business-use-case-usage"])
                # Parser les infos de rate limit
                
            return response.json()
            
        except requests.exceptions.HTTPError as e:
            print(f"❌ Erreur HTTP: {e}")
            return {"error": str(e)}
        except requests.exceptions.RequestException as e:
            print(f"❌ Erreur de requête: {e}")
            return {"error": str(e)}
    
    def search_hashtag(self, hashtag: str) -> Optional[str]:
        """Recherche l'ID d'un hashtag."""
        endpoint = "ig_hashtag_search"
        params = {
            "user_id": self.business_id,
            "q": hashtag
        }
        result = self._make_request(endpoint, params)
        
        if "data" in result and result["data"]:
            return result["data"][0]["id"]
        return None
    
    def get_hashtag_recent_media(self, hashtag_id: str, limit: int = 50) -> List[dict]:
        """Récupère les médias récents d'un hashtag."""
        endpoint = f"{hashtag_id}/recent_media"
        params = {
            "user_id": self.business_id,
            "fields": "id,caption,media_type,timestamp,like_count,comments_count",
            "limit": min(limit, 50)  # Max 50 par requête
        }
        result = self._make_request(endpoint, params)
        return result.get("data", [])
    
    def get_hashtag_top_media(self, hashtag_id: str) -> List[dict]:
        """Récupère les médias top d'un hashtag."""
        endpoint = f"{hashtag_id}/top_media"
        params = {
            "user_id": self.business_id,
            "fields": "id,caption,media_type,timestamp,like_count,comments_count"
        }
        result = self._make_request(endpoint, params)
        return result.get("data", [])

# Initialisation du client
client = InstagramAPIClient(INSTAGRAM_ACCESS_TOKEN, INSTAGRAM_BUSINESS_ID)
print("✅ Client API Instagram configuré")

## 2. Structures de Données pour le Stockage

Définition des structures pour stocker uniquement les métadonnées, signaux statistiques et tendances agrégées.

In [ ]:
@dataclass
class HashtagMetadata:
    """Métadonnées d'un hashtag (sans données personnelles)."""
    hashtag: str
    hashtag_id: str
    collection_date: str
    total_posts_sampled: int
    media_type_distribution: Dict[str, int]
    
@dataclass
class AggregatedStats:
    """Signaux statistiques agrégés."""
    hashtag: str
    period: str  # 'daily', 'weekly', 'monthly'
    avg_likes: float
    avg_comments: float
    median_likes: float
    median_comments: float
    engagement_rate: float
    posting_frequency: float  # posts par heure
    peak_hours: List[int]
    
@dataclass
class TrendSignal:
    """Signal de tendance détecté."""
    hashtag: str
    trend_type: str  # 'rising', 'stable', 'declining'
    confidence_score: float
    velocity: float  # changement de volume
    detection_date: str
    related_hashtags: List[str]
    style_keywords: List[str]
    sentiment_score: float

@dataclass
class TextAnalysisResult:
    """Résultats d'analyse textuelle agrégée."""
    hashtag: str
    top_keywords: Dict[str, int]
    style_categories: Dict[str, float]
    color_mentions: Dict[str, int]
    brand_mentions_count: int  # Count seulement, pas les noms
    avg_caption_length: float
    language_distribution: Dict[str, float]
    emoji_usage_rate: float

print("✅ Structures de données définies")

## 3. Collecte des Hashtags Mode

Liste des hashtags mode populaires à analyser.

In [ ]:
# Hashtags mode à analyser (organisés par catégorie)
FASHION_HASHTAGS = {
    "general": [
        "fashion", "style", "ootd", "outfitoftheday", "fashionblogger",
        "streetstyle", "fashionstyle", "instafashion", "fashionista"
    ],
    "trends": [
        "fashiontrends", "trending", "viral", "aesthetic", "y2k",
        "minimalist", "maximalist", "quiet luxury", "oldmoney"
    ],
    "categories": [
        "streetwear", "vintage", "sustainable fashion", "luxury",
        "casualstyle", "workwear", "athleisure", "bohostyle"
    ],
    "seasonal": [
        "springfashion", "summerstyle", "falloutfit", "winterfashion",
        "transitionalstyle"
    ],
    "colors": [
        "allblack", "neutrals", "colorful", "pastels", "earthtones",
        "monochrome"
    ]
}

# Aplatir la liste pour analyse
all_hashtags = [tag for category in FASHION_HASHTAGS.values() for tag in category]
print(f"📊 {len(all_hashtags)} hashtags à analyser")
print(f"📁 Catégories: {list(FASHION_HASHTAGS.keys())}")

## 4. Analyse des Hashtags Publics

Fonctions pour collecter et analyser les métadonnées des hashtags.

In [ ]:
class HashtagAnalyzer:
    """Analyseur de hashtags pour extraire des tendances agrégées."""
    
    def __init__(self, api_client: InstagramAPIClient):
        self.client = api_client
        self.metadata_store: List[HashtagMetadata] = []
        self.stats_store: List[AggregatedStats] = []
        self.trends_store: List[TrendSignal] = []
        
    def collect_hashtag_data(self, hashtag: str) -> Optional[Dict]:
        """
        Collecte les données d'un hashtag et retourne uniquement les métadonnées agrégées.
        Aucune donnée personnelle n'est stockée.
        """
        print(f"🔍 Analyse du hashtag: #{hashtag}")
        
        # Recherche de l'ID du hashtag
        hashtag_id = self.client.search_hashtag(hashtag)
        if not hashtag_id:
            print(f"  ⚠️ Hashtag non trouvé: {hashtag}")
            return None
        
        # Récupération des médias récents (pour analyse agrégée)
        recent_media = self.client.get_hashtag_recent_media(hashtag_id, limit=50)
        top_media = self.client.get_hashtag_top_media(hashtag_id)
        
        all_media = recent_media + top_media
        
        if not all_media:
            print(f"  ⚠️ Aucun média trouvé pour: {hashtag}")
            return None
        
        # Extraction des métadonnées agrégées (pas de données personnelles)
        metadata = self._extract_metadata(hashtag, hashtag_id, all_media)
        stats = self._compute_aggregated_stats(hashtag, all_media)
        text_analysis = self._analyze_captions(hashtag, all_media)
        
        # Stockage
        self.metadata_store.append(metadata)
        self.stats_store.append(stats)
        
        # Pause pour respecter les rate limits
        time.sleep(1)
        
        return {
            "metadata": asdict(metadata),
            "stats": asdict(stats),
            "text_analysis": text_analysis
        }
    
    def _extract_metadata(self, hashtag: str, hashtag_id: str, media: List[dict]) -> HashtagMetadata:
        """Extrait les métadonnées sans informations personnelles."""
        media_types = Counter(m.get("media_type", "UNKNOWN") for m in media)
        
        return HashtagMetadata(
            hashtag=hashtag,
            hashtag_id=hashtag_id,
            collection_date=datetime.now().isoformat(),
            total_posts_sampled=len(media),
            media_type_distribution=dict(media_types)
        )
    
    def _compute_aggregated_stats(self, hashtag: str, media: List[dict]) -> AggregatedStats:
        """Calcule les statistiques agrégées."""
        likes = [m.get("like_count", 0) for m in media if m.get("like_count")]
        comments = [m.get("comments_count", 0) for m in media if m.get("comments_count")]
        
        # Analyse des heures de publication
        timestamps = []
        for m in media:
            if "timestamp" in m:
                try:
                    dt = datetime.fromisoformat(m["timestamp"].replace("Z", "+00:00"))
                    timestamps.append(dt.hour)
                except:
                    pass
        
        peak_hours = []
        if timestamps:
            hour_counts = Counter(timestamps)
            peak_hours = [h for h, _ in hour_counts.most_common(3)]
        
        return AggregatedStats(
            hashtag=hashtag,
            period="snapshot",
            avg_likes=np.mean(likes) if likes else 0,
            avg_comments=np.mean(comments) if comments else 0,
            median_likes=np.median(likes) if likes else 0,
            median_comments=np.median(comments) if comments else 0,
            engagement_rate=np.mean(likes) / 1000 if likes else 0,  # Estimation
            posting_frequency=len(media) / 24,  # Approximation
            peak_hours=peak_hours
        )
    
    def _analyze_captions(self, hashtag: str, media: List[dict]) -> Dict:
        """
        Analyse textuelle des captions - stocke uniquement des agrégats.
        Aucune caption individuelle n'est conservée.
        """
        captions = [m.get("caption", "") for m in media if m.get("caption")]
        
        if not captions:
            return {}
        
        # Extraction de mots-clés mode (pas les captions complètes)
        fashion_keywords = self._extract_fashion_keywords(captions)
        
        # Détection des couleurs mentionnées
        color_mentions = self._extract_color_mentions(captions)
        
        # Analyse de sentiment agrégée
        sentiments = [TextBlob(c).sentiment.polarity for c in captions[:20]]  # Limite pour perf
        
        # Statistiques textuelles
        avg_length = np.mean([len(c) for c in captions])
        emoji_count = sum(1 for c in captions if any(ord(char) > 127462 for char in c))
        
        return {
            "top_keywords": dict(fashion_keywords.most_common(20)),
            "color_mentions": dict(color_mentions),
            "avg_sentiment": np.mean(sentiments) if sentiments else 0,
            "avg_caption_length": avg_length,
            "emoji_usage_rate": emoji_count / len(captions) if captions else 0,
            "hashtag_co_occurrences": self._extract_cooccurring_hashtags(captions)
        }
    
    def _extract_fashion_keywords(self, captions: List[str]) -> Counter:
        """Extrait les mots-clés liés à la mode."""
        fashion_terms = {
            "style", "outfit", "look", "wear", "fashion", "trend", "aesthetic",
            "vintage", "minimal", "chic", "elegant", "casual", "formal",
            "dress", "shirt", "pants", "jacket", "shoes", "bag", "accessories",
            "summer", "winter", "spring", "fall", "seasonal",
            "sustainable", "ethical", "luxury", "affordable", "designer"
        }
        
        all_words = []
        for caption in captions:
            words = re.findall(r'\b[a-z]+\b', caption.lower())
            fashion_words = [w for w in words if w in fashion_terms or len(w) > 4]
            all_words.extend(fashion_words)
        
        return Counter(all_words)
    
    def _extract_color_mentions(self, captions: List[str]) -> Counter:
        """Extrait les mentions de couleurs."""
        colors = {
            "black", "white", "red", "blue", "green", "yellow", "orange",
            "pink", "purple", "brown", "beige", "grey", "gray", "navy",
            "cream", "nude", "tan", "burgundy", "olive", "coral", "teal"
        }
        
        color_counts = Counter()
        for caption in captions:
            words = set(caption.lower().split())
            for color in colors:
                if color in words:
                    color_counts[color] += 1
        
        return color_counts
    
    def _extract_cooccurring_hashtags(self, captions: List[str]) -> Dict[str, int]:
        """Extrait les hashtags co-occurents (agrégés)."""
        hashtag_pattern = r'#(\w+)'
        all_hashtags = []
        
        for caption in captions:
            hashtags = re.findall(hashtag_pattern, caption.lower())
            all_hashtags.extend(hashtags)
        
        return dict(Counter(all_hashtags).most_common(15))

# Initialisation de l'analyseur
analyzer = HashtagAnalyzer(client)
print("✅ Analyseur de hashtags initialisé")

## 5. Suivi des Volumes de Mentions

Analyse temporelle des volumes et détection de tendances.

In [ ]:
class TrendTracker:
    """Suivi des volumes et détection des tendances."""
    
    def __init__(self):
        self.volume_history: Dict[str, List[Dict]] = {}
        self.trend_signals: List[TrendSignal] = []
        
    def record_volume(self, hashtag: str, volume: int, engagement: float):
        """Enregistre un point de volume pour un hashtag."""
        if hashtag not in self.volume_history:
            self.volume_history[hashtag] = []
        
        self.volume_history[hashtag].append({
            "timestamp": datetime.now().isoformat(),
            "volume": volume,
            "engagement": engagement
        })
    
    def calculate_trend_velocity(self, hashtag: str, window_days: int = 7) -> float:
        """
        Calcule la vélocité d'une tendance (taux de changement).
        Retourne un score entre -1 (déclin) et 1 (croissance).
        """
        if hashtag not in self.volume_history:
            return 0.0
        
        history = self.volume_history[hashtag]
        if len(history) < 2:
            return 0.0
        
        # Calcul simple de la tendance
        recent = history[-1]["volume"]
        older = history[0]["volume"]
        
        if older == 0:
            return 1.0 if recent > 0 else 0.0
        
        change_rate = (recent - older) / older
        return max(-1, min(1, change_rate))  # Normaliser entre -1 et 1
    
    def detect_trend_type(self, velocity: float) -> str:
        """Détermine le type de tendance basé sur la vélocité."""
        if velocity > 0.2:
            return "rising"
        elif velocity < -0.2:
            return "declining"
        else:
            return "stable"
    
    def generate_trend_signal(
        self,
        hashtag: str,
        related_hashtags: List[str],
        style_keywords: List[str],
        sentiment: float
    ) -> TrendSignal:
        """Génère un signal de tendance."""
        velocity = self.calculate_trend_velocity(hashtag)
        trend_type = self.detect_trend_type(velocity)
        
        # Calcul du score de confiance basé sur les données disponibles
        history_length = len(self.volume_history.get(hashtag, []))
        confidence = min(1.0, history_length / 10)  # Plus de données = plus de confiance
        
        signal = TrendSignal(
            hashtag=hashtag,
            trend_type=trend_type,
            confidence_score=confidence,
            velocity=velocity,
            detection_date=datetime.now().isoformat(),
            related_hashtags=related_hashtags[:5],  # Limiter
            style_keywords=style_keywords[:10],
            sentiment_score=sentiment
        )
        
        self.trend_signals.append(signal)
        return signal
    
    def get_rising_trends(self, min_confidence: float = 0.5) -> List[TrendSignal]:
        """Retourne les tendances en hausse avec confiance suffisante."""
        return [
            t for t in self.trend_signals
            if t.trend_type == "rising" and t.confidence_score >= min_confidence
        ]
    
    def export_trends_summary(self) -> pd.DataFrame:
        """Exporte un résumé des tendances en DataFrame."""
        if not self.trend_signals:
            return pd.DataFrame()
        
        return pd.DataFrame([asdict(t) for t in self.trend_signals])

# Initialisation du tracker
trend_tracker = TrendTracker()
print("✅ Tracker de tendances initialisé")

## 6. Simulation de Données (Mode Démo)

Pour tester sans accès API réel, voici des données simulées représentatives.

In [ ]:
def generate_demo_data() -> pd.DataFrame:
    """
    Génère des données de démonstration pour tester les visualisations.
    Ces données simulent les résultats d'une analyse réelle.
    """
    np.random.seed(42)
    
    hashtags = [
        "quietluxury", "oldmoney", "minimalist", "streetwear", "y2k",
        "sustainable", "vintage", "athleisure", "bohostyle", "workwear",
        "casualchic", "aesthetic", "ootd", "fashiontrends", "styleinspo"
    ]
    
    data = []
    for hashtag in hashtags:
        # Simuler des tendances différentes
        if hashtag in ["quietluxury", "oldmoney", "sustainable"]:
            trend = "rising"
            velocity = np.random.uniform(0.3, 0.8)
        elif hashtag in ["y2k", "athleisure"]:
            trend = "declining"
            velocity = np.random.uniform(-0.5, -0.2)
        else:
            trend = "stable"
            velocity = np.random.uniform(-0.15, 0.15)
        
        data.append({
            "hashtag": hashtag,
            "volume_mentions": np.random.randint(10000, 500000),
            "avg_likes": np.random.randint(500, 15000),
            "avg_comments": np.random.randint(20, 500),
            "engagement_rate": np.random.uniform(2, 8),
            "sentiment_score": np.random.uniform(-0.2, 0.8),
            "trend_type": trend,
            "velocity": velocity,
            "confidence": np.random.uniform(0.6, 0.95),
            "peak_hour": np.random.choice([9, 12, 18, 20, 21]),
            "top_color": np.random.choice(["black", "white", "beige", "navy", "brown"]),
            "category": np.random.choice(["luxury", "casual", "streetwear", "vintage", "minimal"])
        })
    
    return pd.DataFrame(data)

# Générer les données de démo
demo_df = generate_demo_data()
print("✅ Données de démonstration générées")
print(f"📊 {len(demo_df)} hashtags analysés")
demo_df.head()

## 7. Visualisation des Tendances

Graphiques pour visualiser les tendances mode détectées.

In [ ]:
# Visualisation 1: Volume des mentions par hashtag
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# 1. Top hashtags par volume
ax1 = axes[0, 0]
top_volume = demo_df.nlargest(10, 'volume_mentions')
colors = ['#2ecc71' if t == 'rising' else '#e74c3c' if t == 'declining' else '#3498db' 
          for t in top_volume['trend_type']]
ax1.barh(top_volume['hashtag'], top_volume['volume_mentions'], color=colors)
ax1.set_xlabel('Volume de mentions')
ax1.set_title('📊 Top 10 Hashtags par Volume')
ax1.invert_yaxis()

# 2. Tendances (rising vs declining)
ax2 = axes[0, 1]
trend_counts = demo_df['trend_type'].value_counts()
colors_pie = ['#2ecc71', '#3498db', '#e74c3c']
ax2.pie(trend_counts.values, labels=trend_counts.index, autopct='%1.1f%%', 
        colors=colors_pie, startangle=90)
ax2.set_title('📈 Distribution des Tendances')

# 3. Engagement vs Sentiment
ax3 = axes[1, 0]
scatter = ax3.scatter(
    demo_df['engagement_rate'], 
    demo_df['sentiment_score'],
    c=demo_df['velocity'],
    cmap='RdYlGn',
    s=demo_df['volume_mentions']/5000,
    alpha=0.7
)
plt.colorbar(scatter, ax=ax3, label='Vélocité')
ax3.set_xlabel('Taux d\'engagement (%)')
ax3.set_ylabel('Score de sentiment')
ax3.set_title('💬 Engagement vs Sentiment')

# Annotations pour les points importants
for _, row in demo_df.iterrows():
    if row['velocity'] > 0.5 or row['velocity'] < -0.3:
        ax3.annotate(f"#{row['hashtag']}", 
                    (row['engagement_rate'], row['sentiment_score']),
                    fontsize=8, alpha=0.8)

# 4. Vélocité des tendances
ax4 = axes[1, 1]
sorted_df = demo_df.sort_values('velocity', ascending=True)
colors_vel = ['#2ecc71' if v > 0 else '#e74c3c' for v in sorted_df['velocity']]
ax4.barh(sorted_df['hashtag'], sorted_df['velocity'], color=colors_vel)
ax4.axvline(x=0, color='gray', linestyle='--', alpha=0.5)
ax4.set_xlabel('Vélocité (taux de changement)')
ax4.set_title('🚀 Vélocité des Tendances')

plt.tight_layout()
plt.savefig('data/processed/trends_overview.png', dpi=150, bbox_inches='tight')
plt.show()

print("✅ Graphiques sauvegardés dans data/processed/")

In [ ]:
# Visualisation 2: Nuage de mots des tendances
# Simulation de mots-clés mode populaires

fashion_keywords = {
    "minimal": 150, "elegant": 120, "chic": 110, "casual": 100,
    "sustainable": 95, "vintage": 90, "luxury": 85, "streetwear": 80,
    "aesthetic": 75, "trendy": 70, "classic": 65, "modern": 60,
    "comfortable": 55, "stylish": 50, "effortless": 45, "timeless": 40,
    "curated": 35, "capsule": 30, "quality": 28, "investment": 25,
    "neutrals": 22, "earth tones": 20, "monochrome": 18, "layered": 15
}

plt.figure(figsize=(12, 6))
wordcloud = WordCloud(
    width=1200, height=600,
    background_color='white',
    colormap='viridis',
    max_words=50,
    relative_scaling=0.5
).generate_from_frequencies(fashion_keywords)

plt.imshow(wordcloud, interpolation='bilinear')
plt.axis('off')
plt.title('☁️ Nuage de Mots-Clés Mode', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.savefig('data/processed/fashion_wordcloud.png', dpi=150, bbox_inches='tight')
plt.show()

## 8. Analyse des Couleurs Tendance

Distribution des couleurs les plus mentionnées dans les descriptions.

In [ ]:
# Analyse des couleurs mentionnées (données simulées)
color_data = {
    "black": 2500,
    "white": 2100,
    "beige": 1800,
    "navy": 1200,
    "brown": 1100,
    "grey": 950,
    "cream": 800,
    "green": 600,
    "burgundy": 450,
    "blue": 400
}

# Couleurs réelles pour la visualisation
color_map = {
    "black": "#1a1a1a", "white": "#f5f5f5", "beige": "#d4b896",
    "navy": "#1e3a5f", "brown": "#8b4513", "grey": "#808080",
    "cream": "#fffdd0", "green": "#2e8b57", "burgundy": "#800020",
    "blue": "#4169e1"
}

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Bar chart des couleurs
colors_list = list(color_data.keys())
values = list(color_data.values())
bar_colors = [color_map[c] for c in colors_list]

ax1.barh(colors_list, values, color=bar_colors, edgecolor='black', linewidth=0.5)
ax1.set_xlabel('Nombre de mentions')
ax1.set_title('🎨 Couleurs les Plus Mentionnées')
ax1.invert_yaxis()

# Pie chart
ax2.pie(values, labels=colors_list, colors=bar_colors, autopct='%1.1f%%',
        startangle=90, explode=[0.05 if i < 3 else 0 for i in range(len(colors_list))])
ax2.set_title('🎨 Distribution des Couleurs')

plt.tight_layout()
plt.savefig('data/processed/color_analysis.png', dpi=150, bbox_inches='tight')
plt.show()

## 9. Export des Tendances Agrégées

Sauvegarde des données pour utilisation ultérieure (uniquement métadonnées et agrégats).

In [ ]:
def export_aggregated_trends(df: pd.DataFrame, output_dir: str = "data/processed"):
    """
    Exporte les tendances agrégées dans différents formats.
    
    IMPORTANT: Seules les données agrégées sont stockées.
    Aucune donnée personnelle, image ou contenu individuel n'est conservé.
    """
    import os
    os.makedirs(output_dir, exist_ok=True)
    
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    
    # 1. Export CSV - Tendances principales
    trends_file = f"{output_dir}/fashion_trends_{timestamp}.csv"
    df.to_csv(trends_file, index=False)
    print(f"✅ Tendances exportées: {trends_file}")
    
    # 2. Export JSON - Structure pour l'API
    trends_json = {
        "metadata": {
            "export_date": datetime.now().isoformat(),
            "total_hashtags_analyzed": len(df),
            "data_type": "aggregated_statistics_only",
            "contains_personal_data": False
        },
        "trends_summary": {
            "rising": df[df['trend_type'] == 'rising']['hashtag'].tolist(),
            "stable": df[df['trend_type'] == 'stable']['hashtag'].tolist(),
            "declining": df[df['trend_type'] == 'declining']['hashtag'].tolist()
        },
        "top_engagement": df.nlargest(5, 'engagement_rate')[['hashtag', 'engagement_rate']].to_dict('records'),
        "color_trends": color_data,
        "style_keywords": fashion_keywords
    }
    
    json_file = f"{output_dir}/fashion_trends_{timestamp}.json"
    with open(json_file, 'w', encoding='utf-8') as f:
        json.dump(trends_json, f, indent=2, ensure_ascii=False)
    print(f"✅ JSON exporté: {json_file}")
    
    # 3. Rapport résumé
    report = f"""
# 📊 Rapport Tendances Mode Instagram
**Date:** {datetime.now().strftime("%d/%m/%Y %H:%M")}

## Résumé
- **Hashtags analysés:** {len(df)}
- **Tendances en hausse:** {len(df[df['trend_type'] == 'rising'])}
- **Tendances stables:** {len(df[df['trend_type'] == 'stable'])}
- **Tendances en déclin:** {len(df[df['trend_type'] == 'declining'])}

## Top Tendances Émergentes
{chr(10).join([f"- #{h}" for h in df[df['trend_type'] == 'rising'].nlargest(5, 'velocity')['hashtag']])}

## Couleurs Dominantes
{chr(10).join([f"- {c}: {v} mentions" for c, v in list(color_data.items())[:5]])}

## Mots-clés Style Populaires
{chr(10).join([f"- {k}" for k in list(fashion_keywords.keys())[:10]])}

---
*Données agrégées uniquement - Aucune donnée personnelle stockée*
"""
    
    report_file = f"{output_dir}/fashion_report_{timestamp}.md"
    with open(report_file, 'w', encoding='utf-8') as f:
        f.write(report)
    print(f"✅ Rapport exporté: {report_file}")
    
    return trends_file, json_file, report_file

# Exporter les données
export_files = export_aggregated_trends(demo_df)
print("\n📁 Fichiers exportés avec succès!")

## 10. Dashboard Récapitulatif

Vue d'ensemble des insights clés.

In [ ]:
# Dashboard récapitulatif
def display_dashboard(df: pd.DataFrame):
    """Affiche un dashboard récapitulatif des tendances."""
    
    print("=" * 60)
    print("📊 DASHBOARD TENDANCES MODE INSTAGRAM")
    print("=" * 60)
    print(f"📅 Date d'analyse: {datetime.now().strftime('%d/%m/%Y %H:%M')}")
    print()
    
    # KPIs principaux
    print("🎯 KPIs PRINCIPAUX")
    print("-" * 40)
    print(f"  📈 Hashtags analysés: {len(df)}")
    print(f"  🚀 Tendances en hausse: {len(df[df['trend_type'] == 'rising'])}")
    print(f"  📉 Tendances en déclin: {len(df[df['trend_type'] == 'declining'])}")
    print(f"  ➡️  Tendances stables: {len(df[df['trend_type'] == 'stable'])}")
    print()
    
    # Top tendances émergentes
    print("🔥 TOP TENDANCES ÉMERGENTES")
    print("-" * 40)
    rising = df[df['trend_type'] == 'rising'].nlargest(5, 'velocity')
    for _, row in rising.iterrows():
        print(f"  • #{row['hashtag']:<20} | Vélocité: +{row['velocity']:.2f} | Engagement: {row['engagement_rate']:.1f}%")
    print()
    
    # Tendances en déclin
    print("📉 TENDANCES EN DÉCLIN")
    print("-" * 40)
    declining = df[df['trend_type'] == 'declining'].nsmallest(3, 'velocity')
    for _, row in declining.iterrows():
        print(f"  • #{row['hashtag']:<20} | Vélocité: {row['velocity']:.2f}")
    print()
    
    # Meilleur engagement
    print("💬 MEILLEUR ENGAGEMENT")
    print("-" * 40)
    top_engagement = df.nlargest(3, 'engagement_rate')
    for _, row in top_engagement.iterrows():
        print(f"  • #{row['hashtag']:<20} | Engagement: {row['engagement_rate']:.1f}% | Sentiment: {row['sentiment_score']:.2f}")
    print()
    
    # Couleurs tendance
    print("🎨 COULEURS TENDANCE (Top 5)")
    print("-" * 40)
    for color, count in list(color_data.items())[:5]:
        bar = "█" * (count // 200)
        print(f"  • {color:<12} {bar} ({count})")
    print()
    
    print("=" * 60)
    print("✅ Analyse terminée - Données agrégées uniquement")
    print("=" * 60)

# Afficher le dashboard
display_dashboard(demo_df)

## 📋 Notes sur la Confidentialité des Données

Ce notebook est conçu pour respecter les règles de confidentialité:

### ✅ Ce qui est stocké:
- **Métadonnées**: IDs de hashtags, dates de collecte, types de média
- **Statistiques agrégées**: Moyennes, médianes, distributions
- **Tendances**: Signaux de tendance, vélocité, scores de confiance
- **Mots-clés**: Termes de mode extraits (pas les captions complètes)

### ❌ Ce qui n'est PAS stocké:
- Images ou médias
- Captions individuelles complètes
- Informations personnelles des utilisateurs
- Noms d'utilisateurs ou profils
- Données permettant l'identification

### 🔒 API Instagram Graph:
- Utilisation de l'API officielle uniquement
- Respect des rate limits
- Aucun scraping de données privées